# ETL with pandas: one GDP table from several sources

This notebook builds a single country-year GDP per capita series from four public datasets.

You will do the three steps of **ETL**:

1. **Extract.** Fetch the files as the publishers ship them.
2. **Transform.** Make the files compatible, stack them, and put the GDP figures on one price scale.
3. **Load.** Average the GDP estimates for each country and year, and save that table.

The first two sources are the **Maddison Project** and **Gapminder**. Two more, the **Penn World Table** and the **World Bank**, come in once that pipeline works. Reusing the same cleaning rules is the point of the second half.

The first run needs a network connection. Later runs read the copies saved under `data/raw/`.

In [1]:
import pandas as pd  # load pandas and call it pd

## 1. What ETL is for

A publisher designs a file for their own project. Column names, country identifiers, year ranges, and units are chosen by that project. A second publisher makes different choices. Both files can be correct and still be impossible to stack.

ETL is the work of turning those files into one table you are willing to analyse.

| Step | Question it answers | In this notebook |
| --- | --- | --- |
| Extract | What did the publisher actually give us? | Download each CSV and look at it unchanged |
| Transform | What would a common shape be? | One row per source, country, and year, with an ISO country code |
| Load | Where does the finished table go? | A CSV of the mean GDP per capita |

The load step is often a database. A CSV is the same idea: a finished table that another person can open without rerunning the cleaning.

## 2. The table we want

The finished table has one row per country and year:

| Column | Meaning |
| --- | --- |
| `country_code` | ISO 3166-1 alpha-3 code |
| `country` | Country name |
| `year` | Year |
| `gdp_per_capita_mean` | Mean of the source estimates |
| `n_sources` | How many sources contributed to that mean |
| `gdp_lowest`, `gdp_highest` | Lowest and highest source after scaling |
| `source_range` | Highest minus lowest. Large values mean the sources disagree |

A mean of one number is just that number. The file we save keeps rows where **at least two sources** have a value.

There are more GDP datasets than the four below.

## 3. Read in the data

#### Gapminder

In [2]:
url = "https://raw.githubusercontent.com/jennybc/gapminder/main/inst/extdata/gapminder.tsv"  # web address of the Gapminder table

df_gap = pd.read_csv(url, sep="\t")  # read that tab-separated file

df_gap.head(5)

,country,continent,year,lifeExp,pop,gdpPercap
0,Afghanistan,Asia,1952,28.801,8425333,779.445314
1,Afghanistan,Asia,1957,30.332,9240934,820.853030
2,Afghanistan,Asia,1962,31.997,10267083,853.100710
3,Afghanistan,Asia,1967,34.020,11537966,836.197138
4,Afghanistan,Asia,1972,36.088,13079460,739.981106


## 4. Exercises

The questions below use the two tables from this notebook: **Maddison** and **Gapminder** (`df_gap`, already loaded in section 3). Run the notebook from the top so `df_gap` exists before you start.

Each question is about **coverage**: which years and which countries are in the file. Leave the GDP columns aside for now. Maddison's `GDP` is a total for the country, and Gapminder's `gdpPercap` is an amount per person, so the two columns are on different scales.

Answer each part by printing the result in the code cell under it.

Maddison is not one file. It is a folder of CSV files, one per country, at `datasets/maddison_by_country/` in this repository. From the `week-06/assignments` folder, that path is `../../datasets/maddison_by_country/`.

Each country file has the columns `Entity`, `Code`, `Year`, and `GDP`.

Nine files in that folder are **not countries**. Eight are regional totals, and the file name contains `(Maddison)`. The ninth is `World.csv`. Leave those nine out. Counting them would treat a region as a country.


### Exercise 1 — Load Maddison into one table

Build one dataframe called `df_mad`.

1. Print `Path.cwd()` so you can see which folder the kernel is reading from. If that path ends in `week-06/assignments`, the Maddison folder is `Path("../../datasets/maddison_by_country")`.
2. List the CSV files in that folder.
3. Skip any file whose name contains `(Maddison)`, and skip `World.csv`.
4. Read each remaining file and stack the tables into `df_mad`.

Then print `df_mad.head()`, `df_mad.columns`, and `df_mad.shape`.


In [3]:
# Read the country CSVs, skip the nine non-country files, and stack them into df_mad.
from pathlib import Path  # Path is used to point at files and folders
print(Path.cwd())

folder = Path('D:/PYTHON/DATA ENGINEERING/Week 05/maddison_by_country')  # folder that holds one CSV per country

folder.glob("*.csv")  # match every CSV file in that folder
print(folder.glob("*.csv"))

dataframes = []  # list that will hold one table per country
for file in folder.glob("*.csv"):  # take each CSV file in turn
    # Skip any file whose name contains (Maddison) or is World.csv
    if "(Maddison)" in file.name or file.name == "World.csv":
        continue
    #print(file)  # unused: would print the file path
    dataframes.append(pd.read_csv(file))  # read the file and add its table to the list

df_mad = pd.concat(dataframes,ignore_index=True)  # stack every country table into one table

df_mad.to_csv("D:/PYTHON/DATA ENGINEERING/Week 05/all_countries.csv")  # save that combined table as a CSV file

d:\PYTHON\DATA ENGINEERING\Week 06


In [4]:
df_mad.head()

,Entity,Code,Year,GDP,GDP (Annotations)
0,Afghanistan,AFG,1950,9421400000,NaN
1,Afghanistan,AFG,1951,9692280000,NaN
2,Afghanistan,AFG,1952,10017325000,NaN
3,Afghanistan,AFG,1953,10630520000,NaN
4,Afghanistan,AFG,1954,10866360000,NaN


In [4]:
df_mad.columns

Index(['Entity', 'Code', 'Year', 'GDP', 'GDP (Annotations)'], dtype='str')

In [5]:
df_mad.shape

(15961, 5)

### Exercise 2 — Maddison: years and countries

Use `df_mad`. The year column is `Year`. The country column is `Entity`.

**A. Year range.** What is the earliest year, and what is the latest year?

**B. Observations per year.** How many rows does each year have? Sort the result by year so the series reads from earliest to latest.

**C. Observations per ten-year block.** Group the rows into decades. The label of a year is `Year // 10 * 10`: 1952 and 1957 both belong to 1950, and 1962 belongs to 1960. How many rows fall in each block?

**D. Countries.** How many distinct countries are in `Entity`?


In [6]:
# A. Earliest and latest year.
print('A) Earliest and Latest year:')
min_year = df_mad['Year'].min()
print('- Earliest year of all rows:',min_year)
max_year = df_mad['Year'].max()
print('- Latest year of all rows  :',max_year)

# B. Rows per year, sorted by year.
obs_per_year = df_mad.groupby('Year').size().reset_index(name='Observations_per_Year')
print('B) Observation per Year:')
print(obs_per_year.head())

# C. Rows per ten-year block. The block label is year // 10 * 10.

obs_per_10year = (df_mad['Year']//10*10).value_counts().sort_index().reset_index(name='Observations_per_10_Years')
print('C) Observation per 10 Years:')
print(obs_per_10year.tail())

# D. Number of distinct countries.
number_of_country = df_mad['Entity'].nunique()
print('D). Number of distinct countries:',number_of_country)

A) Earliest and Latest year:
- Earliest year of all rows: 1
- Latest year of all rows  : 2022
B) Observation per Year:
   Year  Observations_per_Year
0     1                     12
1  1000                      6
2  1500                     11
3  1600                     15
4  1700                     17
C) Observation per 10 Years:
    Year  Observations_per_10_Years
21  1980                       1675
22  1990                       1688
23  2000                       1690
24  2010                       1690
25  2020                        507
D). Number of distinct countries: 169


### Exercise 3 — Gapminder: the same four questions

Repeat Exercise 2 for `df_gap`.

The year column is `year`. The country column is `country`. Gapminder has no regional-total files to skip.

**A.** Earliest year and latest year.

**B.** Rows per year, sorted by year.

**C.** Rows per ten-year block, with the same rule as in Exercise 2.

**D.** Number of distinct countries.


In [7]:
# A–D for df_gap. Columns: "year" and "country".
# A. Earliest and latest year.
print('A) Earliest and Latest year:')
min_year = df_gap['year'].min()
print('- Earliest year of all rows:',min_year)
max_year = df_gap['year'].max()
print('- Latest year of all rows  :',max_year)

# B. Rows per year, sorted by year.
obs_per_year = df_gap.groupby('year').size().reset_index(name='Observations_per_Year')
print('B) Observation per Year:')
print(obs_per_year.head())

# C. Rows per ten-year block. The block label is year // 10 * 10.

obs_per_10year = (df_gap['year']//10*10).value_counts().sort_index().reset_index(name='Observations_per_10_Years')
print('C) Observation per 10 Years:')
print(obs_per_10year.tail())

# D. Number of distinct countries.
number_of_country = df_gap['country'].nunique()
print('D). Number of distinct countries:',number_of_country)

A) Earliest and Latest year:
- Earliest year of all rows: 1952
- Latest year of all rows  : 2007
B) Observation per Year:
   year  Observations_per_Year
0  1952                    142
1  1957                    142
2  1962                    142
3  1967                    142
4  1972                    142
C) Observation per 10 Years:
   year  Observations_per_10_Years
1  1960                        284
2  1970                        284
3  1980                        284
4  1990                        284
5  2000                        284
D). Number of distinct countries: 142


### Exercise 4 — The years the two datasets share

**A. Shared span.** The shared span is the stretch of calendar years that sits inside both ranges. It starts at the **later** of the two earliest years, and it ends at the **earlier** of the two latest years. Print that start year and that end year.

**B. Years present in both tables.** A year can fall inside the shared span and still be missing from one file. Gapminder does not record every year. Print the years that appear as values in **both** `df_mad["Year"]` and `df_gap["year"]`, sorted.


In [48]:
# A. Start and end of the shared span.
# The 2 earliest years:
print('A. Start and end of the shared span:')
if df_mad['Year'].min() > df_gap['year'].min():
    span_start = df_mad['Year'].min()
else:
    span_start = df_gap['year'].min()
print('- Start of the shared span:',span_start)

# The 2 latest years:
if df_mad['Year'].max() < df_gap['year'].max():
    span_end = df_mad['Year'].max()
else:
    span_end = df_gap['year'].max()
print('- End of the shared span  :',span_end)

# B. Years that occur in both tables, sorted.
print('B. Years that occur in both tables, sorted:')

# Turn into sets:
common_years = set(df_mad['Year']) & set(df_gap['year'])
common_years = set(df_mad['Year']).intersection(set(df_gap['year']))

# Sorting
sorted_common_years = sorted(common_years)
# Other option:
import numpy as np
sorted_common_years = np.intersect1d(df_mad['Year'], df_gap['year'])
print(sorted_common_years)

A. Start and end of the shared span:
- Start of the shared span: 1952
- End of the shared span  : 2007
B. Years that occur in both tables, sorted:
[1952 1957 1962 1967 1972 1977 1982 1987 1992 1997 2002 2007]


### Exercise 5 — The countries the two datasets share

Compare the names in `df_mad["Entity"]` with the names in `df_gap["country"]`. The match is on the **spelling of the name**. `"South Korea"` and `"Korea, Rep."` are different strings, so they stay in different lists.

**A.** Which names appear in both tables? Print them in alphabetical order.

**B.** How many names is that?

**C.** Print the names that appear in only one table. From that list, write down three pairs that you think are the same place under two spellings. A short comment in the code cell is enough. Leave the names as they are in the files.


In [ ]:
# A. Names in both tables, sorted.
print('A. Names in both tables:')
common_names = set(df_mad['Entity']) & set(df_gap['country'])
# Sorting
sorted_common_names = sorted(common_names)
print(sorted_common_names)

# B. How many shared names.
print('B. Number of shared names:', len(common_names))

# C. Names in only one table, and three likely spelling pairs.
unique_names = set(df_mad['Entity']) ^ set(df_gap['country'])
sorted_unique_names = sorted(unique_names)
print(sorted_unique_names)

# Three likely spelling pairs for the same place:
# 1. 'Sudan' vs 'Sudan (former)'
# 2. 'Congo', 'Congo, Dem. Rep.', 'Congo, Rep.' vs "Democratic Republic of Congo"
# 3. "Korea, Rep.", 'Korea, Dem. Rep.', 'Korea, Rep.' vs "South Korea"
# 4. 'Yemen' vs 'Yemen, Rep.'


A. Names in both tables:
['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina', 'Australia', 'Austria', 'Bahrain', 'Bangladesh', 'Belgium', 'Benin', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Costa Rica', "Cote d'Ivoire", 'Croatia', 'Cuba', 'Denmark', 'Djibouti', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Ethiopia', 'Finland', 'France', 'Gabon', 'Gambia', 'Germany', 'Ghana', 'Greece', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Haiti', 'Honduras', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Jordan', 'Kenya', 'Kuwait', 'Lebanon', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Malaysia', 'Mali', 'Mauritania', 'Mauritius', 'Mexico', 'Mongolia', 'Montenegro', 'Morocco', 'Mozambique', 'Myanmar', 'Namibia', 'Nepal'

### What to turn in

Leave the printed results in this notebook. A reader should be able to find:

- for Maddison and for Gapminder: the earliest year, the latest year, the rows per year, the rows per decade, and the number of countries
- the shared span, as a start year and an end year
- the years that occur in both tables
- the country names that occur in both tables, how many that is, and three spelling pairs
